# Reproducing Severson et al. (2017) from dandiset 000226

**Paper.** Severson KS, Xu D, Van de Loo M, Bai L, Ginty DD, O'Connor DH (2017). Active touch and
self-motion encoding by Merkel cell-associated afferents. *Neuron* 94:666-676.
[doi:10.1016/j.neuron.2017.03.045](https://doi.org/10.1016/j.neuron.2017.03.045). This attempt used
the full text of the accepted author manuscript in PubMed Central (PMC5528144) and the publisher's
Supplemental Information (Table S1 and the supplemental figure legends). The version of record is
not open access. The paper's analysis code is not public; the Methods say it is available on
request, and no repository for it was found.

**Dandiset.** [DANDI:000226](https://dandiarchive.org/dandiset/000226/0.230607.1747), version
0.230607.1747. The paper cites no DANDI version (it predates the deposit). This is the only
published version; the draft has the same 60 assets and size, so the most recent published
version is pinned.

**What the dandiset holds.** One NWB file per recording: 56 trigeminal afferents recorded with
high-speed whisker video, and 4 EMG sessions, which match the 60 recordings of Table S1. Each file
holds the spike times of one unit (none in three EMG files) and, for every 5 s trial, the whisker
kinematics, contact, distance to the pole, curvature change, and for most recordings with a pole,
forces and bending moment, at 500 Hz. The files do not say which experiment a recording belongs to,
the afferent type, or which afferents were identified as Merkel afferents; these come from Table S1
(see `table-s1-link`).

**Streaming.** Each file is read whole into memory from the archive and never written to disk
(see `merkel.py`). The run takes about 21 minutes, most of it reading 13.7 GB and fitting the spline models.

| Claim | What | Status |
|---|---|---|
| table-s1-link | Recordings match Table S1 | tested |
| video-frames | Frame counts (54.6 M, 65.3 M) | tested |
| merkel-minutes | Minutes of Merkel recordings | tested (outside labels) |
| fig2c-baseline-rate | Non-whisking rate 0.0 Hz | tested |
| fig2c-touch-rate | Contact rate 48.0 Hz, all touch-responsive | tested |
| fig2c-wt-star | 13 WT* afferents, whisking rate 19.1 Hz | tested |
| fig3a-direction-selective | Merkel and SA contact responses direction selective | tested (outside labels) |
| fig3d-gam-full | Full model r = 0.71 | tested |
| fig3d-gam-single | M0 alone 0.45, M0' alone 0.42 | tested |
| fig3d-gam-best-pair | M0 and M0' the best pair (0.63) | tested |
| fig4c-viscoelastic | Viscoelastic model comparable to the M0, M0' model | tested (outside labels) |
| fig5-phase-modulation | WT* phase modulation 2.1 against 51.2 Hz | tested |
| fig5e-phase-span | Preferred phases span the cycle | tested |
| figs3-phase-invariance | Preferred phase invariant to amplitude, frequency, setpoint | tested |
| figs6-kinematic-gam | Kinematic models predict phase tuning (r = 0.97) | tested on 15 of 28 |
| fig7g-phase-span-28 | 28 WT* phases, more in retraction | blocked |
| fig6-cut-eliminated | Cutting eliminated responses in 6 of 13 | blocked |
| fig6-cut-remained | Responses remained in 7 of 13 | blocked |
| fig6a-final-length | Final whisker length 0.37 mm | blocked |
| fig7d-cut-emg | Cut-whisker tuning matches EMG (r = 0.66) | blocked |
| fig7e-lost-rate-kinematics | Lost rate matches acceleration or jerk (r = 0.71) | blocked |
| fig7f-precut-emg | Pre-cut afferents match EMG (r = 0.76) | blocked |
| figs4-nasolabialis | Nasolabialis motor units follow negative velocity | tested |
| figs4-intrinsic-protractor | Intrinsic protractor EMG follows positive velocity | blocked |
| sigma-kappa | Curvature noise 7.8% | tested |
| figs1-optotag-latency | Light-evoked spike latency 4.9 ms | blocked |
| figs1-labelling-specificity | 99.2% of labelled endings are Merkel | blocked |
| contact-classifier | Contact classifier 99.5% correct | blocked |

In [ ]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import merkel

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
table = pd.DataFrame(merkel.table_s1()).set_index("rec")
assets = merkel.assets()
print(f"dandiset {merkel.DANDISET_ID} version {merkel.DANDISET_VERSION}: {len(assets)} assets, "
      f"{sum(a[1] for a in assets) / 1e9:.2f} GB")

The recordings to keep frames for are chosen from Table S1 before loading: the touch recordings
(for the touch models) and the 15 WT* recordings of Figure S3 (for the whisking-in-air models).
Their identifiers are matched to the files in `table-s1-link`; here only the file name is needed,
through the identifier read while loading.

In [ ]:
WT_STAR_15 = sorted(list(table.index[(table.group == "touch") & (table.cls == "WT*")]) +
                    list(table.index[table.group == "whiskonly"]))
TOUCH_33 = list(table.index[table.group == "touch"])

import re
def rec_of(identifier):
    # KS0240A -> 240A; KS0007A -> 7A; KS0125_ -> 125_ (matched by number in table-s1-link)
    m = re.fullmatch(r"KS0*(\d+)([A-Z_])", identifier)
    return f"{int(m.group(1))}{m.group(2)}"

def number_of(identifier):
    return int(re.fullmatch(r"KS0*(\d+)[A-Z_]", identifier).group(1))

# The touch and WT* recordings by mouse and date, to decide what to keep before the identifier is read.
def wanted(path):
    sub = path.split("/")[0].replace("sub-", "")
    date = path.split("ses-")[1][:8]
    rows = table[(table.mouse == sub) & (table.date.map(lambda d: pd.to_datetime(d, format="%m/%d/%y").strftime("%Y%m%d")) == date)]
    return set(rows.index)

def reduce_one(item):
    i, (path, size, url) = item
    cands = wanted(path)
    return merkel.reduce(path, url, seed=1000 + i,
                         keep_touch=bool(cands & set(TOUCH_33)), keep_wia=bool(cands & set(WT_STAR_15)))

with ThreadPoolExecutor(3) as ex:
    recs = list(ex.map(reduce_one, enumerate(assets)))
print(f"loaded and reduced {len(recs)} files in {(time.time() - started) / 60:.1f} min")

## Data check

In [ ]:
info = pd.DataFrame([{
    "path": r["path"].split("/")[1][:48], "identifier": r["identifier"], "subject": r["subject_id"], "sex": r["sex"],
    "strain": r["strain"], "date": r["session_start"][:10], "trials": r["n_trials"],
    "spikes": r.get("n_spikes"), "rate_hz": r["rates"], "frames_per_series": r["lengths"],
    "trial_dur_s": r["trial_duration"], "MOI": r["MOI_values"][:2], "MOI_nan": r["MOI_nan"],
    "cutting": r["cutting_groups"], "has_M0": "M0" in r["series_present"],
    "has_dpole": "distanceToPoleCenter" in r["series_present"]} for r in recs])
print("Files:", len(info), "; subjects:", info.subject.nunique(), "; files with spike times:", int(info.spikes.notna().sum()))
print("Archive summary: 60 files, 43 subjects (assetsSummary).")
print("Series per file at", sorted({x for r in recs for x in r["rates"]}), "Hz with", sorted({x for r in recs for x in r["lengths"]}), "frames per trial")
print("Trial table columns:", sorted({tuple(r["trial_columns"]) for r in recs}))
print("Acquisition groups present in any file:", sorted({x for r in recs for x in r["acquisition"]}))
print("Cutting groups in any file:", sorted({x for r in recs for x in r["cutting_groups"]}))
print("Trial durations in the trial table (s):", sorted({tuple(np.round(r["trial_duration"], 3)) for r in recs}),
      "; series start equals trial start (max offset s):", max(r["start_offset"] for r in recs))
print("related_publications:", sorted({p for r in recs for p in r["related_publications"]}))
print("Clock times of session_start_time:", sorted({r["session_start"][11:19] for r in recs}))
print()
print(info.sort_values("path").to_string(index=False))

Each 5 s trial holds 2,500 frames at 500 Hz, as the Methods state, but `stop_time` in every trial
table is 7 s after `start_time`. Every trial's series start at the trial's `start_time`. All trials
of all files are labelled `intact` in `cuttingGroup`, including the 13 progressive cutting
recordings, and `MOI` (moment of inertia) has one value per file; one file (KS0253A) has `MOI`
missing in every trial. Session start times are all at 12:00, so only their dates carry
information. The archive's count of 43 subjects matches the files.

In [ ]:
units = {}
for r in recs:
    for v, u in r["units"].items():
        units.setdefault(v, set()).add(u)
print("Stated units:", {v: sorted(u) for v, u in units.items()})
rng_tab = pd.DataFrame([{"variable": v,
                         "min": min(r["ranges"][v][0] for r in recs if v in r["ranges"]),
                         "max": max(r["ranges"][v][1] for r in recs if v in r["ranges"]),
                         "files": sum(v in r["ranges"] for r in recs),
                         "finite_fraction_min": min(r["ranges"][v][2] for r in recs if v in r["ranges"])}
                        for v in merkel.VARIABLES])
print(rng_tab.to_string(index=False))
missing = {r["identifier"]: [v for v in ("M0", "Faxial", "distanceToPoleCenter") if v not in r["series_present"]] for r in recs}
print("\nFiles without forces and moment:", sorted(k for k, v in missing.items() if "M0" in v))
print("Files without distance to pole:", sorted(k for k, v in missing.items() if "distanceToPoleCenter" in v))

Phase spans -pi to pi radians, amplitude is in degrees, and contact is 0 or 1. The ranges include
extreme values (position from -985 to 596 degrees, frequency from -250 to 250 Hz, M0 down to
-0.52 N m) that the Glossary's outlier removal excludes. The unit of `distanceToPoleCenter` is
"unknown"; the next cells check it against contact, and the units of the derivatives.

In [ ]:
spk = [r for r in recs if r["has_spikes"]]
place = pd.DataFrame([{"identifier": r["identifier"], **r["spike_placement"], "sorted": r["spikes_sorted"]} for r in spk])
print("Spikes in the 2,500 frames of a trial:", place.in_frames.sum(), "; inside a trial interval but after its frames:",
      place.in_trial_after_frames.sum(), "; outside every trial:", place.outside_trials.sum())
print("Files whose spike times are not sorted:", list(place.identifier[~place.sorted]))
sig = pd.DataFrame([{"identifier": r["identifier"], "pos": r["signature"][0], "spk": r["signature"][1]} for r in recs])
dup_pos = sig[sig.duplicated("pos", keep=False)]
dup_spk = sig[sig.duplicated("spk", keep=False) & sig.identifier.isin([r["identifier"] for r in spk])]
print("Files with identical whisker position in their first 5 trials:", list(dup_pos.identifier) or "none")
print("Files with identical spike trains:", list(dup_spk.identifier) or "none")
same_session = info.groupby(["subject", "date"]).identifier.apply(list)
print("Sessions with more than one file (different units; checked above for duplicates):",
      [v for v in same_session if len(v) > 1])

No two files share their whisker traces or spike trains, so no recording is deposited twice; the
seven sessions with two files hold two different units each, as Table S1 lists them. Spike times
in one file are not sorted (named above). No spike falls outside the 5 s of video of its trial.

In [ ]:
# distanceToPoleCenter against contact, and the sign of M0 against the direction of motion at contact onset.
rows = []
for r in recs:
    if "episodes" not in r:
        continue
    ep = r["episodes"]
    ok = np.isfinite(ep[:, 0]) & (ep[:, 3] != 0)
    rows.append({"identifier": r["identifier"], "contacts": len(ep),
                 "M0_sign_matches_velocity_at_onset": np.mean(np.sign(ep[ok, 0]) == np.sign(ep[ok, 3]))})
sgn = pd.DataFrame(rows)
print("Per touch recording, fraction of contacts whose mean M0 has the sign of the whisker velocity at contact onset:")
print(sgn.M0_sign_matches_velocity_at_onset.describe().round(3).to_string())
secs = pd.DataFrame([{"identifier": r["identifier"], **{k: round(v / 60, 1) for k, v in r["category_seconds"].items()}} for r in recs])
print("\nMinutes per category over all files:", secs.drop(columns="identifier").sum().round(0).to_dict())
dp = pd.DataFrame([{"identifier": r["identifier"], "contact_p1": r["dpole_contact_pct"][0], "contact_p50": r["dpole_contact_pct"][1],
                    "contact_p99": r["dpole_contact_pct"][2], "free_p1": (r["dpole_free_pct"] or [np.nan])[0],
                    "free_p50": (r["dpole_free_pct"] or [np.nan, np.nan])[1]} for r in recs if "dpole_contact_pct" in r])
print("\ndistanceToPoleCenter in frames with contact = 1 (percentiles 1, 50, 99) and without contact (1, 50), per file:")
print(dp.round(2).to_string(index=False))
kp = pd.DataFrame({"phase": recs[0]["kinematics_by_phase"]["centres"].round(2),
                   **{f"{q}_{k}": np.mean([r["kinematics_by_phase"][k] for r in recs if "kinematics_by_phase" in r], axis=0)
                      for k in ("position",) for q in ("mean",)},
                   "mean_velocity": np.mean([r["kinematics_by_phase"]["velocity"] for r in recs if "kinematics_by_phase" in r], axis=0)})
print("\nMean position (minus its mean, degrees) and velocity (as stored; per ms, see the ratio check below) by phase, whisking frames, averaged over files:")
print(kp.round(2).to_string(index=False))
neg_vel = [np.mean(r["kinematics_by_phase"]["velocity"][r["kinematics_by_phase"]["centres"] < 0]) for r in recs if "kinematics_by_phase" in r]
print("Files where the mean velocity over phases below 0 is negative:", int(np.sum(np.array(neg_vel) < 0)), "of", len(neg_vel))
pdir = pd.DataFrame([r["phase_direction"] for r in recs])
for k in ("sin", "cos"):
    print(f"Correlation of {k}(stored phase) with {k}(Hilbert phase of band-passed position), per file: median",
          round(float(pdir[k].median()), 4), ", range", round(float(pdir[k].min()), 4), "to", round(float(pdir[k].max()), 4))
print("(sin -1 and cos +1: the stored phase is the negative of the Hilbert phase; both -1: shifted by pi)")
print("Files where mean position is highest in the bin nearest phase 0:",
      int(sum(np.argmin(np.abs(r["kinematics_by_phase"]["centres"])) in np.argsort(r["kinematics_by_phase"]["position"])[-2:]
              for r in recs if "kinematics_by_phase" in r)), "of", sum("kinematics_by_phase" in r for r in recs),
      "(bins of width pi/6; the two bins either side of 0 count)")
ratios = pd.DataFrame([{"identifier": r["identifier"], **r["derivative_ratios"]} for r in recs])
print("\nStored derivative over the change of its variable per 2 ms frame (0.5 means per ms, 500 per s):")
print(ratios.drop(columns="identifier").describe().loc[["count", "min", "50%", "max"]].round(3).to_string())

Frames with contact = 1 have distances near 0 and frames without contact have larger ones (table
above), consistent with the Methods' d_pole in mm, measured to the pole's edge, though the files do
not state the unit. Protraction (positive velocity) contacts mostly have M0 > 0, as Figure 3F has
it, so the sign of M0 gives the contact direction.

Mean position is highest in the bins next to phase 0 in all 60 files, so phase 0 is full
protraction, as in the Hilbert convention the paper cites (Hill et al. 2011). But the stored phase
runs backwards in time: the sine of the stored phase is anticorrelated (median -0.9999) and its
cosine correlated (median 0.9999) with those of the Hilbert phase computed here from the
band-passed position, so the stored phase is the negative of the Hilbert phase, not a shift by pi.
Velocity is negative at phases below 0 in every file. In the files, phases from -pi to 0 are
retraction and 0 to pi protraction, the reverse of the usual convention. Neither the files nor the
paper state which convention the stored phase follows.

The stored velocity, acceleration, jerk and the rates of change of moment and forces are per
millisecond (ratio near 0.5 to the change per 2 ms frame), while the files state per second
("degree/s", "N*m/s"). Phase curves and the spline models do not depend on these scales; the
viscoelastic model is given dM0 in N m per second.

In [ ]:
# Clock: responses with the true spike times against spikes shifted by 0.5 s within each trial.
clk = pd.DataFrame([{"identifier": r["identifier"], "rec": rec_of(r["identifier"]),
                     "contact_minus_wia_true": r["clock"]["true"]["contact_minus_wia"],
                     "contact_minus_wia_shifted": r["clock"]["shifted"]["contact_minus_wia"],
                     "phase_depth_true": r["clock"]["true"]["phase_depth"],
                     "phase_depth_shifted": r["clock"]["shifted"]["phase_depth"]} for r in spk])
clk["touch_contrast_holds"] = clk.contact_minus_wia_true > 2 * np.abs(clk.contact_minus_wia_shifted)
clk["phase_contrast_holds"] = clk.phase_depth_true > 1.5 * clk.phase_depth_shifted
clk["established"] = clk.touch_contrast_holds.fillna(False) | clk.phase_contrast_holds
print(clk.round(2).to_string(index=False))
print("\nClock shown by one of the two contrasts:", int(clk.established.sum()), "of", len(clk))
print("Not established:", list(clk.rec[~clk.established]))

The spikes and the video series share a clock when the touch response (contact minus whisking in
air) or the phase modulation depth is clearly larger with the true spike times than with spikes
shifted by 0.5 s. Recordings with no touch response and little phase tuning cannot show it either
way; they are named above.

In [ ]:
# Progressive cutting recordings: whisking-in-air rate over the session.
prog = [r for r in spk if rec_of(r["identifier"]) in set(table.index[table.group == "progressive"])]
print("Progressive cutting recordings:", len(prog), "; cutting groups in their trial tables:",
      sorted({g for r in prog for g in r["cutting_groups"]}), "; MOI values per file:", sorted({len(r["MOI_values"]) for r in prog}))
print("(The rate by fifth of the session is shown under fig6-cut-eliminated.)")

The clock is shown for 53 of 57 recordings with spikes. For four touch recordings with little
whisking-in-air activity (named above), neither contrast is clear, because contacts last long enough
that spikes shifted by 0.5 s still fall in contact; their timing is taken as it is.

## table-s1-link

> The dandiset's recordings can be matched one to one to the recordings of Table S1, and the
> matched records agree on mouse, sex, mouse line and date.

In [ ]:
rows = []
for r in recs:
    rec = rec_of(r["identifier"])
    if rec in table.index:
        match, how = rec, "id"
    else:
        cands = [t for t in table.index if re.fullmatch(r"\d+", t[:-1]) and int(t[:-1]) == number_of(r["identifier"])
                 and table.loc[t, "mouse"] == r["subject_id"]]
        match, how = (cands[0], "number and mouse") if len(cands) == 1 else (None, "none")
    row = {"identifier": r["identifier"], "matched": match, "how": how}
    if match:
        t = table.loc[match]
        norm = lambda x: x.replace(" ", "").lower()
        row.update({"mouse_ok": t.mouse == r["subject_id"], "sex_ok": t.sex == r["sex"],
                    "line_ok": norm(t.line) == norm(r["strain"]),
                    "date_ok": pd.to_datetime(t.date, format="%m/%d/%y").strftime("%Y-%m-%d") == r["session_start"][:10],
                    "group": t.group})
    rows.append(row)
link = pd.DataFrame(rows)
link["all_ok"] = link[["mouse_ok", "sex_ok", "line_ok", "date_ok"]].all(axis=1)
REC = dict(zip(link.identifier, link.matched))
print(link.to_string(index=False))
n = len(link)
print(f"\nmatched: {link.matched.notna().sum()} of {n}; distinct: {link.matched.nunique()}; by number and mouse only: {(link.how == 'number and mouse').sum()}")
print(f"agree on mouse, sex, line and date: {link.all_ok.sum()} of {n} ({100 * link.all_ok.mean():.1f}%)")
print("Table S1 recordings without a file:", sorted(set(table.index) - set(link.matched.dropna())))
print("Files per Table S1 group:", link.group.value_counts().to_dict())

In [ ]:
# The transcription against Table S1 parsed from the Supplemental Information PDF, fetched from the publisher.
supp, unparsed, nbytes = merkel.supplement_table_s1()
print(f"supplement: {nbytes / 1e6:.1f} MB; rows parsed {len(supp)}, row-like lines not parsed {unparsed}")
print("rows per block:", pd.Series([x["group"] for x in supp]).value_counts().to_dict())
diff, only_ours, only_supp = merkel.compare_table_s1(merkel.table_s1(), supp)
print("rows only in the transcription:", only_ours, "; only in the supplement:", only_supp)
print("recordings that differ, per column:", diff)
# Negative control: the comparison finds a change made to a copy of the transcription.
changed = [dict(x) for x in merkel.table_s1()]
changed[0]["cls"], changed[10]["merkel"] = "T", not changed[10]["merkel"]
print("with two values changed in a copy:", {k: v for k, v in merkel.compare_table_s1(changed, supp)[0].items() if v})


**Result: reproduced.** All 60 files match a distinct Table S1 recording, 58 by recording id and 2 (KS0125_ and KS0128_, which lack the unit letter) by number and mouse, and all 60 agree on mouse, sex, mouse line and date. The files' trial and unit tables hold none of Table S1's other columns (afferent type, Merkel identification, class, experiment group), so these come from the table. The transcription in `merkel.py` agrees with the table parsed from the publisher's supplement on all 60 rows and every column used, including the group; the comparison finds both values changed in a test copy.

In [ ]:
for r in recs:
    r["rec"] = REC[r["identifier"]]
by_rec = {r["rec"]: r for r in recs}
def group(g):
    return [by_rec[x] for x in table.index[table.group == g]]
touch = group("touch")
print("Touch recordings:", len(touch), "; types:", table.loc[[r["rec"] for r in touch]].type.value_counts().to_dict(),
      "; identified Merkel:", int(table.loc[[r["rec"] for r in touch]].merkel.sum()))

## video-frames

> The initial dataset of 33 touch afferents comprises 54,647,500 video frames with simultaneous
> single-unit recordings, and the paper analyses 65,257,500 frames (2,175 min) in total.

In [ ]:
frames = lambda rs: sum(r["n_trials"] for r in rs) * merkel.FRAMES_PER_TRIAL
analysed = group("touch") + group("cutonly") + group("whiskonly") + group("progressive")
afferents = analysed + group("excluded")
readings = {"33 touch recordings": frames(touch),
            "(a) 53 analysed afferents": frames(analysed),
            "(b) 53 afferents and 4 EMG sessions": frames(analysed + group("emg")),
            "(c) 56 afferent recordings": frames(afferents),
            "(d) all 60 files": frames(recs)}
for k, v in readings.items():
    print(f"{k:40s} {v:>12,d} frames = {v / 500 / 60:7.1f} min")
print(f"\nPaper: 54,647,500 (tolerance 5,464,750) and 65,257,500 (tolerance 6,525,750)")
print("touch within tolerance:", abs(readings["33 touch recordings"] - 54647500) <= 5464750)
for k in list(readings)[1:]:
    print(f"total {k}: within tolerance {abs(readings[k] - 65257500) <= 6525750}, ratio {readings[k] / 65257500:.3f}")

**Result: reproduced.** The 33 touch recordings hold 54,660,000 frames (paper 54,647,500). For the total, the 53 analysed afferents, reading (a), hold 65,710,000 frames or 2,190 min (paper 65,257,500 and 2,175 min); readings (b) to (d) are also within the 10% tolerance, at 1.042 to 1.084 times the paper's value.

## merkel-minutes

> Identified Merkel afferents account for 823 minutes of the initial dataset and 846 minutes
> (25,365,000 frames) overall, against 999 minutes for unidentified afferents in the initial
> dataset.

In [ ]:
minutes = lambda rs: sum(r["n_trials"] for r in rs) * 5 / 60
is_m = lambda r: bool(table.loc[r["rec"], "merkel"])
m_touch = minutes([r for r in touch if is_m(r)])
u_touch = minutes([r for r in touch if not is_m(r)])
m_a = minutes([r for r in analysed if is_m(r)])
m_b = minutes([r for r in afferents if is_m(r)])
print(f"Merkel, touch recordings: {m_touch:.1f} min ({sum(is_m(r) for r in touch)} afferents); paper 823 (tolerance 82.3)")
print(f"Unidentified, touch recordings: {u_touch:.1f} min; paper 999 (tolerance 99.9)")
print(f"Merkel overall, (a) 17 analysed: {m_a:.1f} min; (b) all 20 identified: {m_b:.1f} min; paper 846 (tolerance 84.6)")
print("within tolerance:", abs(m_touch - 823) <= 82.3, abs(u_touch - 999) <= 99.9, abs(m_a - 846) <= 84.6, abs(m_b - 846) <= 84.6)

**Result: reproduced.** The 14 Merkel touch recordings hold 823.8 min (paper 823) and the unidentified touch recordings 998.2 min (paper 999). The 17 analysed Merkel afferents, reading (a), hold 846.4 min (paper 846); all 20 identified Merkel afferents hold 922.9 min, which is also within tolerance.

## fig2c-baseline-rate, fig2c-touch-rate and fig2c-wt-star

> During non-whisking periods most afferents spiked at low rates (0.0 plus or minus 0.2 Hz, median
> and IQR, n = 33). All afferents responded with increased spike rates during whisker-pole contact
> (48.0 plus or minus 96.3 Hz). A subset of 13 of the 33 touch afferents (WT*) spiked especially
> strongly during whisking in air (19.1 plus or minus 58.2 Hz), and the others divide into WT and T.

In [ ]:
def iqr(x):
    q = np.percentile(x, [25, 75])
    return q[1] - q[0]
cr = pd.DataFrame([{"rec": r["rec"], "type": table.loc[r["rec"], "type"], "merkel": table.loc[r["rec"], "merkel"],
                    "table_class": table.loc[r["rec"], "cls"],
                    "nonwhisking": r["category_rates"]["nonwhisking"]["rate"],
                    "wia": r["category_rates"]["wia"]["rate"], "contact": r["category_rates"]["contact"]["rate"],
                    "nw_min": r["category_rates"]["nonwhisking"]["seconds"] / 60,
                    "wia_min": r["category_rates"]["wia"]["seconds"] / 60,
                    "contact_min": r["category_rates"]["contact"]["seconds"] / 60,
                    "total_spikes": r["total_spikes"], "wia_spikes": r["wia_spikes"],
                    "class_a": merkel.classify({**r["category_rates"], "total_spikes": r["total_spikes"]}),
                    "class_b": merkel.classify({**r["category_rates"], "total_spikes": r["wia_spikes"]}),
                    "class_poisson": merkel.classify({**r["category_rates"], "total_spikes": r["total_spikes"]}, ci=("plo", "phi")),
                    } for r in touch])
print(cr.round(2).to_string(index=False))
print(f"\nnon-whisking rate: median {cr.nonwhisking.median():.2f} Hz, IQR {iqr(cr.nonwhisking):.2f} (paper 0.0 and 0.2); range {cr.nonwhisking.min():.2f} to {cr.nonwhisking.max():.2f}")
print(f"contact rate: median {cr.contact.median():.1f} Hz, IQR {iqr(cr.contact):.1f} (paper 48.0 and 96.3); range {cr.contact.min():.1f} to {cr.contact.max():.1f}")
print(f"  within the claims' tolerances (the paper's IQRs): baseline {abs(cr.nonwhisking.median() - 0.0) <= 0.2}, contact {abs(cr.contact.median() - 48.0) <= 96.3}")
print(f"  within 10% of 48.0 (not the criterion): {abs(cr.contact.median() - 48.0) <= 4.8}")
print("Touch-sensitive (bootstrap CI over trials):", int((cr.class_a != "none").sum()), "of", len(cr),
      "; with Poisson CIs:", int((cr.class_poisson != "none").sum()))
print("Not touch-sensitive:", list(cr.rec[cr.class_a == "none"]))

**Result: reproduced.** The median non-whisking rate over the 33 touch afferents is 0.03 Hz with an IQR of 0.20 Hz (paper 0.0 and 0.2). One afferent, 221B, fires at 34.4 Hz without whisking.

**Result: reproduced.** The median contact rate over the 33 touch afferents is 47.9 Hz with an IQR of 94.8 Hz (paper 48.0 and 96.3), within the claim's tolerance (the paper's IQR) and also within 10% of the paper's median, and all 33 are touch-sensitive by the Glossary rule, with bootstrap or with Poisson confidence intervals.

In [ ]:
for reading, col in (("(a) all spikes", "class_a"), ("(b) whisking-in-air spikes", "class_b"), ("Poisson CIs, (a)", "class_poisson")):
    star = cr[cr[col] == "WT*"]
    agree = (cr[col].str.replace("none", "T") == cr.table_class).sum()
    print(f"{reading}: classes {cr[col].value_counts().to_dict()}; WT* n = {len(star)}, whisking-in-air rate median "
          f"{star.wia.median():.1f} Hz IQR {iqr(star.wia):.1f}; class agrees with Table S1 for {agree} of 33")
for reading, c in (("(a)", "class_a"), ("(b)", "class_b")):
    star = cr[cr[c] == "WT*"]
    print(f"WT* under {reading} by type:", star.type.value_counts().to_dict(), "; Merkel:", int(star.merkel.sum()),
          "; same set as Table S1's WT*:", set(star.rec) == set(cr.rec[cr.table_class == "WT*"]))
print("(paper: 5 Merkel, 7 SA, 1 RA)")
print("\nDisagreements with Table S1 under (a) or (b):")
print(cr[(cr.class_a.str.replace("none", "T") != cr.table_class) | (cr.class_b.str.replace("none", "T") != cr.table_class)][["rec", "table_class", "class_a", "class_b", "wia", "nonwhisking", "contact", "total_spikes", "wia_spikes"]].round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 3))
o = cr.sort_values("contact").reset_index(drop=True)
for col, c in (("nonwhisking", "0.7"), ("wia", "0.3"), ("contact", "tab:purple")):
    ax.plot(o.index, o[col] + 0.01, "o", color=c, label=col)
ax.set_yscale("log"); ax.set_xticks(o.index, o.rec, rotation=90, fontsize=7); ax.set_ylabel("rate + 0.01 (Hz)"); ax.legend(fontsize=7)
print("Plotted rates span", round(cr[["nonwhisking", "wia", "contact"]].min().min(), 3), "to", round(cr[["nonwhisking", "wia", "contact"]].max().max(), 1), "Hz; none outside the axes.")
plt.show()

**Result: reproduced.** Under reading (b), counting the spikes during whisking in air, 13 afferents are WT*, the same 13 as in Table S1, with a median whisking-in-air rate of 19.0 Hz and an IQR of 57.4 Hz (paper 13, and 19.1 plus or minus 58.2). The class agrees with Table S1 for 31 of 33; 223B and 243B are WT in Table S1 and T here. Under reading (a), all spikes, 15 afferents are WT* (median 7.1 Hz) and 29 of 33 agree: 227A and 145B pass the 1 Hz rule (1.05 and 1.26 Hz) and have more than 1,000 spikes in all but fewer than 1,000 during whisking in air.

## fig3a-direction-selective

> Merkel and SA afferents responded to contact in a dramatically direction-selective manner, far
> more strongly to contacts in either the protraction or the retraction direction.

In [ ]:
rows = []
for r in touch:
    ep = r["episodes"]
    ok = np.isfinite(ep[:, 0])
    pro, ret = ep[ok & (ep[:, 0] > 0)], ep[ok & (ep[:, 0] < 0)]
    rp = pro[:, 1].sum() / (pro[:, 2].sum() * merkel.DT) if len(pro) else np.nan
    rr = ret[:, 1].sum() / (ret[:, 2].sum() * merkel.DT) if len(ret) else np.nan
    rows.append({"rec": r["rec"], "type": table.loc[r["rec"], "type"], "merkel": table.loc[r["rec"], "merkel"],
                 "n_protraction": len(pro), "n_retraction": len(ret), "rate_protraction": rp, "rate_retraction": rr,
                 "index": abs(rp - rr) / (rp + rr) if rp + rr > 0 else np.nan,
                 "preferred": "protraction" if rp > rr else "retraction"})
ds = pd.DataFrame(rows)
print(ds.round(2).to_string(index=False))
sa = ds[ds.type == "SA"]
for thr in (0.33, 0.5, 0.67):
    print(f"Merkel and SA with index >= {thr}: {(sa['index'] >= thr).sum()} of {len(sa)}")
print(f"Merkel and SA: median index {sa['index'].median():.2f}; Merkel only: {sa[sa.merkel]['index'].median():.2f} (n = {sa.merkel.sum()}); "
      f"RA: median {ds[ds.type == 'RA']['index'].median():.2f}, {(ds[ds.type == 'RA']['index'] >= 0.5).sum()} of {(ds.type == 'RA').sum()} at 0.5 or more")
print("Preferred direction among Merkel and SA:", sa.preferred.value_counts().to_dict())

**Result: partially reproduced.** 17 of the 26 Merkel and SA afferents (65%) have a direction index of 0.5 or more, short of the 80% the criterion asks; 21 of 26 have 0.33 or more (preferred rate at least twice the other). The median index is 0.65 over Merkel and SA afferents and 0.59 over the 14 Merkel afferents, against 0.23 for RA afferents, none of which reaches 0.5. 18 of the 26 prefer protraction contacts.

## fig3d-gam-full, fig3d-gam-single and fig3d-gam-best-pair

> A full statistical model of ten mechanical and kinematic variables strongly predicted ongoing
> spike rate during touch (r = 0.71 plus or minus 0.16, n = 33). Models fitted to moment or its rate
> of change alone explained spiking relatively poorly (r = 0.45 and 0.42). M0 and its rate of change
> together explained spiking (r = 0.63) better than any other pair of variables, and better than a
> model of M0, Fax and Flat (r = 0.48).

The models are fitted on contact frames during whisking without outliers on any of the ten
variables, at most 60,000 per recording, chosen at random when loading.

In [ ]:
V = merkel.TOUCH_VARS
col = {v: i for i, v in enumerate(V)}
models = {"full": V, "M0": ["M0"], "M0'": ["dM0"], "M0, M0'": ["M0", "dM0"], "M0, Fax, Flat": ["M0", "Faxial", "Flateral"]}
pairs = merkel.model_subsets(V, 2)
fit_rows, fails = [], []
t0 = time.time()
for r in touch:
    X, y, sm = r["touch"]["X"], r["touch"]["y"], r["touch"]["smooth"]
    row = {"rec": r["rec"], "frames": len(y), "clean_contact_frames": r["contact_frames_clean"], "spikes": int(y.sum())}
    for name, vs in list(models.items()) + [(" + ".join(merkel.PAPER_NAME[v] for v in p), list(p)) for p in pairs]:
        if name in row:
            continue
        rr, m = merkel.fit_r(X[:, [col[v] for v in vs]], y, sm)
        row[name] = rr
        if not m.converged:
            fails.append((r["rec"], name))
    fit_rows.append(row)
gam = pd.DataFrame(fit_rows)
print(f"{len(touch)} recordings, {len(models) + len(pairs) - 1} models each, in {(time.time() - t0) / 60:.1f} min; "
      f"fits that did not converge: {len(fails)} {fails[:5]}")
print("Frames fitted per recording: median", int(gam.frames.median()), ", min", gam.frames.min(), "; clean contact frames: median",
      int(gam.clean_contact_frames.median()), ", min", gam.clean_contact_frames.min())
summ = gam[list(models)].agg(["mean", "std", "median", "min", "max"]).T
print(summ.round(3).to_string())

In [ ]:
print(f"full: mean r {gam['full'].mean():.3f} (paper 0.71, tolerance 0.16)")
print(f"M0 alone: {gam['M0'].mean():.3f} (paper 0.45, tolerance 0.19); M0' alone: {gam[chr(77) + '0' + chr(39)].mean():.3f} (paper 0.42, tolerance 0.21)")
pair_names = [" + ".join(merkel.PAPER_NAME[v] for v in p) for p in pairs]
pm = gam[pair_names].mean().sort_values(ascending=False)
print("\nPairs by mean r (all 45):")
print(pm.round(3).to_string())
rank = list(pm.index).index("M0 + M0'") + 1
print(f"\nM0 + M0' is ranked {rank} of 45, mean r {pm['M0 + M0' + chr(39)]:.3f} (paper 0.63, tolerance 0.20); "
      f"M0, Fax, Flat: {gam['M0, Fax, Flat'].mean():.3f} (paper 0.48, tolerance 0.19)")
best = pm.index[0]
d = gam["M0 + M0'"] - gam[best]
print(f"M0 + M0' minus the best pair ({best}) per recording: median {d.median():.3f}, M0 + M0' higher in {(d > 0).sum()} of 33")
d2 = gam["M0, M0'"] - gam["M0, Fax, Flat"]
print(f"M0, M0' minus M0, Fax, Flat: mean {d2.mean():.3f}, higher in {(d2 > 0).sum()} of 33, Wilcoxon p = {stats.wilcoxon(d2).pvalue:.2g}")
by_type = gam.assign(type=[table.loc[x, "type"] + ("-Merkel" if table.loc[x, "merkel"] else "") for x in gam.rec]).groupby("type")[["full", "M0", "M0'", "M0, M0'", "M0, Fax, Flat"]].mean()
print("\nMean r by afferent type (Table S1):"); print(by_type.round(3).to_string())

In [ ]:
# Variant: a stronger smoothing penalty (lambda = 100) for the five named models, to show how the choice affects r.
rows = []
for r in touch:
    X, y, sm = r["touch"]["X"], r["touch"]["y"], r["touch"]["smooth"]
    rows.append({name: merkel.fit_r(X[:, [col[v] for v in vs]], y, sm, lam=100.0)[0] for name, vs in models.items()})
var = pd.DataFrame(rows)
print("Mean r with lambda = 100:", var.mean().round(3).to_dict())
print("Mean r with lambda = 1:  ", gam[list(models)].mean().round(3).to_dict())

In [ ]:
# Variants of the 45-pair ranking: lambda = 100 on the same frames, and lambda = 1 fitted on a random half
# of each recording's frames and evaluated on the other half (generator of this cell).
rng_pairs = np.random.default_rng(3)
pv = {"lambda 100": [], "held out": []}
t0 = time.time()
for r in touch:
    X, y, sm = r["touch"]["X"], r["touch"]["y"], r["touch"]["smooth"]
    train = rng_pairs.random(len(y)) < 0.5
    for key, kw in (("lambda 100", {"lam": 100.0}), ("held out", {"train": train})):
        pv[key].append({name: merkel.fit_r(X[:, [col[v] for v in p]], y, sm, **kw)[0] for name, p in zip(pair_names, pairs)})
print(f"{len(touch)} recordings x 45 pairs x 2 variants in {(time.time() - t0) / 60:.1f} min")
for key, rows in [("lambda 1, fitted frames (main)", None)] + list(pv.items()):
    tab = gam[pair_names] if rows is None else pd.DataFrame(rows)
    m = tab.mean().sort_values(ascending=False)
    best_other = m.drop("M0 + M0'").index[0]
    dd = tab["M0 + M0'"] - tab[best_other]
    print(f"\n{key}: M0 + M0' ranked {list(m.index).index(chr(77) + '0 + M0' + chr(39)) + 1} of 45, mean r {m[chr(77) + '0 + M0' + chr(39)]:.3f}; "
          f"best other pair {best_other} {m[best_other]:.3f}; M0 + M0' higher in {(dd > 0).sum()} of 33")
    print("  top five:", m.head(5).round(3).to_dict())

**Result: reproduced.** The full model's mean r over the 33 touch afferents is 0.708 with SD 0.141 (paper 0.71 plus or minus 0.16). With a 100 times stronger penalty it is 0.687. All fits converged.

**Result: reproduced.** M0 alone gives a mean r of 0.441 (paper 0.45) and M0' alone 0.415 (paper 0.42).

**Result: partially reproduced.** M0 and M0' together give a mean r of 0.622 (paper 0.63) and beat the model of M0, Fax and Flat (0.475; paper 0.48) in 31 of 33 afferents (Wilcoxon p = 4.8e-08). But the pair of M0 and whisker velocity (omega) has a slightly higher mean r, 0.633, so M0 and M0' rank second of the 45 pairs; per afferent, M0 and M0' is higher in 19 of 33, with a median difference of 0.003. The rank does not depend on the penalty or on evaluating on the fitted frames: M0 and M0' is also second, behind M0 and omega, with lambda 100 (0.589 against 0.600, higher in 17 of 33) and on held-out frames (0.620 against 0.630, higher in 20 of 33), and the same five pairs lead in all three.

## fig4c-viscoelastic

> A simple viscoelastic model of moment and its rate of change predicted spike rates of Merkel and
> SA afferents during touch at levels comparable to the statistical model based on M0 and its rate
> of change (25 of 26 afferents fitted).

In [ ]:
rng_visco = np.random.default_rng(4)
vis = []
for r in touch:
    if table.loc[r["rec"], "visco"] != "Y":
        continue
    X, sm = r["touch"]["X"], r["touch"]["smooth"]
    dm0_per_s = X[:, col["dM0"]] * 1000  # stored per ms (data check)
    p, loss, ok, scale, at_bound = merkel.fit_visco(X[:, col["M0"]], dm0_per_s, sm / merkel.DT, rng_visco)
    pred = merkel.visco_predict(p, X[:, col["M0"]] / scale, dm0_per_s / scale)
    rv = np.corrcoef(pred, sm)[0, 1] if pred.std() > 0 else np.nan
    vis.append({"rec": r["rec"], "merkel": table.loc[r["rec"], "merkel"], "r_visco": rv,
                "r_gam_M0_M0'": gam.set_index("rec").loc[r["rec"], "M0, M0'"], "converged": ok, "at_bound": ",".join(n for n, b in zip(["k", "C", "eps_lim", "E", "eta"], at_bound) if b),
                "k": p[0], "C": p[1], "eps_lim": p[2], "E": p[3], "eta": p[4]})
vis = pd.DataFrame(vis)
print(vis.round(3).to_string(index=False))
d = vis.r_visco - vis["r_gam_M0_M0'"]
print(f"\n{len(vis)} afferents (Table S1 fitted; 128D excluded); optimizer reported success for {vis.converged.sum()}")
print(f"mean r: viscoelastic {vis.r_visco.mean():.3f}, M0 and M0' model {vis['r_gam_M0_M0' + chr(39)].mean():.3f}; difference {d.mean():.3f} "
      f"(95% bootstrap CI {np.percentile([rng_visco.choice(d, len(d)).mean() for _ in range(2000)], [2.5, 97.5]).round(3)})")
print(f"viscoelastic higher in {(d > 0).sum()} of {len(d)}; Merkel only: mean difference {d[vis.merkel].mean():.3f} (n = {vis.merkel.sum()})")
print("Fits with a parameter at a bound of the fit (as fitted, eta in scaled units):",
      {c: int(vis.at_bound.str.split(",").apply(lambda x: c in x).sum()) for c in ["k", "C", "eps_lim", "E", "eta"]},
      "; fits with any:", int((vis.at_bound != "").sum()), "of", len(vis))
print("Different parameter sets can give nearly the same prediction, so the parameter values above are not interpreted.")
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.plot(vis["r_gam_M0_M0'"], vis.r_visco, "o"); ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("r, spline model of M0 and M0'"); ax.set_ylabel("r, viscoelastic model"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
print("Points outside the axes:", int(((vis.r_visco < 0) | (vis["r_gam_M0_M0'"] < 0)).sum()))
plt.show()

In [ ]:
# Stability: the same fits with three other sets of random starts. The fitted values moved between runs of
# unchanged code and data, so the spread over starts is shown here.
rows = []
for seed in (40, 41, 42):
    g = np.random.default_rng(seed)
    rv = []
    for r in touch:
        if table.loc[r["rec"], "visco"] != "Y":
            continue
        X, sm = r["touch"]["X"], r["touch"]["smooth"]
        dm0_per_s = X[:, col["dM0"]] * 1000
        p, loss, ok, scale, at_bound = merkel.fit_visco(X[:, col["M0"]], dm0_per_s, sm / merkel.DT, g)
        pred = merkel.visco_predict(p, X[:, col["M0"]] / scale, dm0_per_s / scale)
        rv.append({"rec": r["rec"], "r": np.corrcoef(pred, sm)[0, 1] if pred.std() > 0 else np.nan, "bound": bool(at_bound.any())})
    rv = pd.DataFrame(rv).set_index("rec")
    dd = rv.r - vis.set_index("rec")["r_gam_M0_M0'"]
    change = (rv.r - vis.set_index("rec").r_visco).abs()
    rows.append({"start seed": seed, "mean r visco": rv.r.mean(), "mean difference": dd.mean(),
                 "visco higher": int((dd > 0).sum()), "fits at a bound": int(rv.bound.sum()),
                 "largest change in r from the main fit": change.max(), "recordings changed by > 0.01": int((change > 0.01).sum())})
print(pd.DataFrame(rows).round(3).to_string(index=False))


**Result: reproduced.** Over the 25 Merkel and SA afferents, the viscoelastic model's mean r is 0.656 against 0.686 for the spline model of M0 and M0', a difference of -0.030 (95% bootstrap CI -0.061 to -0.006), within the 0.1 the criterion allows; it is higher for 10 of 25. The optimizer reported success for all 25, but 6 of 25 fits end with a parameter at a bound (eps_lim in 4, E in 1, eta in 2), so some fits may be local optima, which would only lower the viscoelastic r. With three other sets of random starts the mean difference is -0.037, -0.040 and -0.017, and 5 to 7 fits end at a bound; single recordings change by up to 0.555 in r, so individual fits are not stable, but the mean difference stays within the criterion. Different parameter sets can give nearly the same prediction, so the fitted parameter values are not interpreted.

## fig5-phase-modulation, fig5e-phase-span and figs3-phase-invariance

> All 15 WT* afferents were dramatically modulated by whisk phase, with rates at non-preferred and
> preferred phases of 2.1 plus or minus 3.4 and 51.2 plus or minus 105.7 Hz (median and IQR).
> Preferred phase spanned the whisk cycle, and was largely invariant across whisk cycles of
> different amplitudes, frequencies and setpoints.

In [ ]:
wt15 = [by_rec[x] for x in WT_STAR_15]
pt = pd.DataFrame([{"rec": r["rec"], "type": table.loc[r["rec"], "type"], "merkel": table.loc[r["rec"], "merkel"],
                    "group": table.loc[r["rec"], "group"],
                    "nonpreferred": r["phase_tuning"][0].min(), "preferred": r["phase_tuning"][0].max(),
                    "pref_phase": r["phase_tuning"][1][np.argmax(r["phase_tuning"][0])],
                    "wia_min": r["category_seconds"]["wia"] / 60} for r in wt15])
pt["ratio"] = pt.preferred / pt.nonpreferred.clip(lower=1e-9)
print(pt.round(2).to_string(index=False))
print(f"\nnon-preferred: median {pt.nonpreferred.median():.2f} Hz, IQR {iqr(pt.nonpreferred):.2f} (paper 2.1 and 3.4)")
print(f"preferred: median {pt.preferred.median():.1f} Hz, IQR {iqr(pt.preferred):.1f} (paper 51.2 and 105.7)")
print(f"within the claim's tolerances (the paper's IQRs, 3.4 and 105.7): {abs(pt.nonpreferred.median() - 2.1) <= 3.4}, {abs(pt.preferred.median() - 51.2) <= 105.7}")
print(f"within 10% of the paper's medians (not the criterion): {abs(pt.nonpreferred.median() - 2.1) <= 0.21}, {abs(pt.preferred.median() - 51.2) <= 5.12}")
print(f"preferred at least twice non-preferred: {(pt.ratio >= 2).sum()} of 15; ratio range {pt.ratio.min():.1f} to {pt.ratio.max():.0f}")

**Result: reproduced.** Over the 15 WT* afferents, the median rate at the non-preferred phase is 2.43 Hz with an IQR of 3.16 Hz (paper 2.1 plus or minus 3.4), and at the preferred phase 51.1 Hz with an IQR of 96.6 Hz (paper 51.2 plus or minus 105.7); both medians are within the claim's tolerances, the paper's IQRs. Every afferent fires at least 9.5 times more at its preferred phase than at its non-preferred phase. Against a 10% band, which is not the criterion, the non-preferred median would be outside (16% above the paper's) and the preferred median inside.

In [ ]:
ph = pt.pref_phase.values
quarters = np.floor((ph + np.pi) / (np.pi / 2)).clip(0, 3).astype(int)
R = np.abs(np.mean(np.exp(1j * ph)))
n = len(ph)
z = n * R ** 2
p_rayleigh = np.exp(np.sqrt(1 + 4 * n + 4 * (n ** 2 - (n * R) ** 2)) - (1 + 2 * n))
print("Preferred phases (rad):", np.round(np.sort(ph), 2))
print("Quarter cycles occupied:", sorted(set(quarters)), "counts", np.bincount(quarters, minlength=4))
print(f"Rayleigh: R = {R:.3f}, z = {z:.2f}, p = {p_rayleigh:.3f}")
vel_at_pref = np.array([r["velocity_by_tuning_bin"][np.argmax(r["phase_tuning"][0])] for r in wt15])
print(f"Preferred phase in (0, pi): {(ph > 0).sum()} of {n}; in (-pi, 0]: {(ph <= 0).sum()}")
print(f"Mean whisker velocity at the preferred phase negative (retraction): {(vel_at_pref < 0).sum()} of {n}")
def rayleigh(ph):
    n, R = len(ph), np.abs(np.mean(np.exp(1j * ph)))
    return R, np.exp(np.sqrt(1 + 4 * n + 4 * (n ** 2 - (n * R) ** 2)) - (1 + 2 * n))
def quarters_of(ph):
    return np.bincount(np.floor((ph + np.pi) / (np.pi / 2)).clip(0, 3).astype(int), minlength=4)
# Variants of the preferred-phase estimator (the criterion uses the bin of highest rate, 30 bins, above).
ph_cm = np.array([np.angle(np.sum(r["phase_tuning"][0] * np.exp(1j * r["phase_tuning"][1]))) for r in wt15])
variants = {"circular mean of the 30-bin tuning curve": ph_cm}
for nb_ in (20, 60):   # highest bin with other bin counts, on the outlier-free whisking-in-air frames kept for figs6
    variants[f"highest of {nb_} bins (outlier-free frames)"] = np.array(
        [(lambda tc, c: c[np.argmax(tc)])(*merkel.phase_tuning(r["wia_frames"]["phase"], r["wia_frames"]["y"], nb_)[:2]) for r in wt15])
for k, v in variants.items():
    Rv, pv_ = rayleigh(v)
    print(f"variant, {k}: quarters {quarters_of(v)}, Rayleigh R = {Rv:.3f}, p = {pv_:.3f}; "
          f"median circular difference from the main estimate {np.median(merkel.circ_diff(v, ph)):.2f} rad")
fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(3, 3))
ax.hist(ph, bins=12, range=(-np.pi, np.pi)); plt.show()

**Result: partially reproduced.** The preferred phases of the 15 WT* afferents fall in all four quarter cycles (7, 4, 1 and 3 afferents), but they are not uniform: Rayleigh R = 0.448, p = 0.047, just below 0.05. The outcome rests on the criterion's estimator (the bin of highest rate of 30) and sits at the threshold: with the circular mean of the tuning curve, p = 0.062, which would be reproduced; with the highest of 20 or 60 bins, p = 0.023 and 0.046. All four quarters are occupied under every estimator. At the preferred phase the whisker is retracting (mean velocity negative) for 11 of 15.

In [ ]:
inv = pd.DataFrame([{"rec": r["rec"], **{v: merkel.circ_diff(*r["phase_halves"][v]) for v in ("amplitude", "frequency", "setpoint")}} for r in wt15])
print(inv.round(2).to_string(index=False))
print("\nMedian absolute circular difference between halves (rad):", inv[["amplitude", "frequency", "setpoint"]].median().round(3).to_dict(),
      "; pi/4 =", round(np.pi / 4, 3))
print("Afferents with a difference above pi/4:", {v: int((inv[v] > np.pi / 4).sum()) for v in ("amplitude", "frequency", "setpoint")})

**Result: reproduced.** The median absolute difference between the preferred phases of the two halves is 0.37 rad for amplitude, 0.59 rad for frequency and 0.48 rad for setpoint, all below pi/4 (0.785). For 4, 6 and 5 of the 15 afferents the difference exceeds pi/4.

## figs6-kinematic-gam

> Statistical models fitted to whisker position, velocity, acceleration and jerk during whisking in
> air predicted phase tuning curves nearly perfectly (r = 0.97 plus or minus 0.04, n = 28), while
> single kinematic variables performed poorly.

Tested on the 15 WT* afferents of Figure S3; the intact condition of the 13 progressive cutting
recordings cannot be separated (see fig6-cut-eliminated). Each model is fitted on 60,000 frames
chosen at random (generator of this section) and predicts every kept frame.

In [ ]:
rng_s6 = np.random.default_rng(6)
K = merkel.KINEMATIC_VARS
combos = [c for k in (1, 2, 3, 4) for c in merkel.model_subsets(K, k)]
names = [", ".join(merkel.PAPER_NAME[v] for v in c) for c in combos]
rows, nfail = [], 0
t0 = time.time()
for r in wt15:
    w = r["wia_frames"]
    X, y, phase = w["X"], w["y"], w["phase"]
    tc_obs, _, b = merkel.phase_tuning(phase, y)
    sub = rng_s6.choice(len(y), min(60000, len(y)), replace=False)
    row = {"rec": r["rec"], "frames": len(y), "spikes": int(y.sum())}
    for c, name in zip(combos, names):
        idx = [K.index(v) for v in c]
        m = merkel.SplineModel().fit(X[sub][:, idx], y[sub])
        nfail += not m.converged
        pred = m.predict(X[:, idx]) * 1000
        row[name] = np.corrcoef(merkel.bin_means(pred, b), tc_obs)[0, 1]
    rows.append(row)
s6 = pd.DataFrame(rows)
print(f"{len(s6)} afferents x {len(combos)} models in {(time.time() - t0) / 60:.1f} min; fits not converged: {nfail}")
print(s6[["rec", "frames", "spikes"] + names].round(3).to_string(index=False))
m6 = s6[names].agg(["mean", "std", "median", "min"]).T.sort_values("mean")
print(); print(m6.round(3).to_string())
full = ", ".join(merkel.PAPER_NAME[v] for v in K)
print(f"\nfull model: mean r {s6[full].mean():.3f}, SD {s6[full].std():.3f} (paper 0.97 and 0.04)")
for v in K:
    print(f"single {merkel.PAPER_NAME[v]}: mean r {s6[merkel.PAPER_NAME[v]].mean():.3f}; lower than full by {s6[full].mean() - s6[merkel.PAPER_NAME[v]].mean():.3f}")

**Result: reproduced.** On the 15 WT* afferents, the model of position, velocity, acceleration and jerk predicts phase tuning curves with a mean r of 0.975 and SD 0.013 (paper 0.97 plus or minus 0.04, on 28). Single variables give mean r from 0.524 to 0.724, at least 0.25 below the full model, and the pair of velocity and acceleration already reaches 0.960. The 13 progressive cutting recordings are not included.

## fig6-cut-eliminated, fig6-cut-remained, fig7g-phase-span-28 (blocked)

> Progressive whisker cutting eliminated spiking during whisking in air in 6 of 13 afferents, and
> in 7 of 13 self-motion responses remained.

Every trial of the 13 progressive cutting recordings is labelled `intact`, and `MOI` has one value
per file, so the cutting conditions cannot be told apart. The rate by fifth of each session shows
that the files hold more than the intact condition.

In [ ]:
fifths = pd.DataFrame([{"rec": r["rec"], "trials": r["n_trials"], "MOI values": len(r["MOI_values"]),
                        "cutting groups": r["cutting_groups"],
                        **{f"fifth {i + 1}": round(x, 2) for i, x in enumerate(r["wia_rate_fifths"])}} for r in prog])
print("Whisking-in-air rate (Hz) by fifth of the session, in trial order:")
print(fifths.to_string(index=False))
f = fifths[[f"fifth {i}" for i in range(1, 6)]].values
print(f"\nlast fifth below 10% of the first: {(f[:, 4] < 0.1 * f[:, 0]).sum()} of {len(f)}; "
      f"last fifth below half the first: {(f[:, 4] < 0.5 * f[:, 0]).sum()} of {len(f)}")

**Result: blocked (metadata insufficient).** Every trial of the 13 progressive cutting recordings is labelled intact and each file has one moment of inertia, so the cutting conditions cannot be told apart. The whisking-in-air rate falls over most of these sessions: the last fifth is below half the first in 10 of 13 and below 10% of it in 5 of 13, which fits the files holding the cut conditions under the intact label.

## What the archive holds for the other blocked claims

The archive's own description of each asset (`variableMeasured`), and every group and dataset in
each file (per-trial series collapsed to one entry), checked for the data the blocked claims need:
EMG, optogenetic stimulation, whisker length and manual contact labels.

In [ ]:
vm = merkel.asset_variables()
print("variableMeasured across assets:", pd.Series([tuple(v) for v in vm.values()]).value_counts().to_dict())
for x in ("7A", "8A", "9A", "11A"):
    r = by_rec[x]
    print(f"{x} ({r['identifier']}): variableMeasured {vm[r['path']]}; neurodata types {r['neurodata_types']}")
    print("   paths outside processing/behavior, general and specifications:",
          [p for p in r["paths"] if not p.startswith(("processing/behavior/", "general", "specifications"))])
    print("   series in processing/behavior:", r["series_present"])
other = sorted({p for r in recs for p in r["paths"] if not p.startswith(("processing/behavior/", "specifications"))})
print("\nPaths outside processing/behavior and specifications in any file:", other)
print("Members of acquisition, analysis or stimulus in any file:",
      sorted({p for r in recs for p in r["paths"] if p.startswith(("acquisition/", "analysis/", "stimulus/presentation/", "stimulus/templates/"))}) or "none")
print("Neurodata types in any file:", sorted({t for r in recs for t in r["neurodata_types"]}))
print("Series in processing/behavior in any file:", sorted({x for r in recs for x in r["series_present"]}))
words = ("emg", "optogen", "stim", "laser", "light", "length", "curat", "manual", "label")
print("Paths or series names containing", words, ":",
      sorted({p for r in recs for p in r["paths"] + r["series_present"] if any(w in p.lower() for w in words)}) or "none")
print("Per-trial size (MB): EMG files", {x: round(by_rec[x]["bytes"] / by_rec[x]["n_trials"] / 1e6, 3) for x in ("7A", "8A", "9A", "11A")},
      "; progressive files, median", round(float(np.median([r["bytes"] / r["n_trials"] / 1e6 for r in prog])), 3))

In [ ]:
# The EMG files' undescribed Mi and dMi series, by phase (30 equal-count bins of whisking frames),
# against positive and negative velocity, to see whether either holds the intrinsic protractor EMG.
rows = []
for x in ("7A", "8A", "9A", "11A"):
    e = by_rec[x].get("extra_by_phase")
    if e is None:
        rows.append({"rec": x, "note": "no Mi/dMi or too few whisking frames"})
        continue
    row = {"rec": x}
    for v in merkel.EXTRA_VARS:
        row[f"{v}_finite"] = e[v + "_finite"]
        row[f"{v}_min"], row[f"{v}_max"] = e[v + "_range"]
        row[f"{v}_curve_sd"] = float(np.std(e[v]))
        for d in ("pos", "neg"):
            row[f"r_{v}_vel_{d}"] = np.corrcoef(e[v], e["vel_" + d])[0, 1] if np.std(e[v]) > 0 else np.nan
    rows.append(row)
print("stored units:", {v: sorted({r["units"].get(v) for r in recs if v in r["units"]}) for v in merkel.EXTRA_VARS})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
allr = pd.DataFrame([{"rec": r["rec"], "group": table.loc[r["rec"], "group"],
                      **{f"r_{v}_vel_pos": np.corrcoef(r["extra_by_phase"][v], r["extra_by_phase"]["vel_pos"])[0, 1]
                         for v in merkel.EXTRA_VARS}}
                     for r in recs if "extra_by_phase" in r and min(np.std(r["extra_by_phase"][v]) for v in merkel.EXTRA_VARS) > 0])
print("\nFor comparison, the same correlations in every file with whisking, median by group:")
print(allr.groupby("group")[["r_Mi_vel_pos", "r_dMi_vel_pos"]].agg(["median", "count"]).round(3).to_string())


**Result: blocked (data missing) for figs4-intrinsic-protractor, fig7d-cut-emg, fig6a-final-length, figs1-optotag-latency, figs1-labelling-specificity and contact-classifier.** The archive lists `BehavioralTimeSeries` and `ProcessingModule` for every asset and `Units` for 57; the intrinsic protractor files 7A, 8A and 9A have no `Units`, and their per-trial size (0.31 MB) equals that of 11A and of the progressive cutting files. No file has a member in acquisition, analysis or stimulus, or a series or path named for EMG, optogenetic stimulation, light, whisker length or manual labels. The trial tables hold MOI, cuttingGroup and the bar position only. The EMG files hold the tracked whisker series that the afferent files hold, including `Mi` (unit N*m) and `dMi` (N*m/s), with values below 3e-8 in absolute size. Their phase curves do not follow positive velocity: r with positive velocity is -0.026 to 0.002 for Mi and -0.570 to -0.534 for dMi in 7A, 8A and 9A, close to the medians in every group of afferent files (-0.026 to -0.014 for Mi, -0.536 to -0.485 for dMi), so neither series is the intrinsic protractor EMG. fig7d-cut-emg also needs the cut condition (see fig6-cut-eliminated).

## fig7f-precut-emg (blocked): the nasolabialis curve alone

> In 5 afferents recorded with the whisker already cut near its base, phase tuning was strongly
> correlated with intrinsic protractor or nasolabialis EMG (r = 0.76 plus or minus 0.06).

The best match is chosen between the intrinsic protractor and nasolabialis curves, and the
intrinsic protractor EMG is not in the dandiset, so the claim stays blocked. The correlation with
the nasolabialis curve alone (11A) is a lower bound on each afferent's best match. Both curves are
on whisking frames in 30 equal-count phase bins; the 11A curve is interpolated (circularly) to each
afferent's bin centres.

In [ ]:
w11 = by_rec["11A"]["whisk_phase_curves"]
o11 = np.argsort(w11["centres"])
rows = []
for r in group("cutonly"):
    w = r["whisk_phase_curves"]
    naso = np.interp(w["centres"], w11["centres"][o11], w11["rate"][o11], period=2 * np.pi)
    rows.append({"rec": r["rec"], "whisking_min": r["category_seconds"]["whisking"] / 60, "spikes": r["n_spikes"],
                 "rate_min": w["rate"].min(), "rate_max": w["rate"].max(),
                 "r_nasolabialis": np.corrcoef(w["rate"], naso)[0, 1],
                 "r_own_neg_velocity": np.corrcoef(w["rate"], w["vel_neg"])[0, 1],
                 "r_own_pos_velocity": np.corrcoef(w["rate"], w["vel_pos"])[0, 1],
                 "contact_flag_frames": r["contact_flag_frames"], "has_dpole": "distanceToPoleCenter" in r["series_present"],
                 "contact_min": r["category_seconds"]["contact"] / 60})
f7 = pd.DataFrame(rows)
print(f7.round(3).to_string(index=False))
print(f"\nmean r with the nasolabialis curve {f7.r_nasolabialis.mean():.3f} (SD {f7.r_nasolabialis.std():.3f}); "
      f"paper's mean best-match r 0.76 plus or minus 0.06, which is at least this")
f7["r_best_own_velocity"] = f7[["r_own_neg_velocity", "r_own_pos_velocity"]].max(axis=1)
print(f"Not the criterion: the better of each afferent's correlation with its own positive and negative velocity curves "
      f"(Figure S4 relates the two muscles to these) has mean {f7.r_best_own_velocity.mean():.3f}, range "
      f"{f7.r_best_own_velocity.min():.3f} to {f7.r_best_own_velocity.max():.3f}; positive velocity is the better for "
      f"{(f7.r_own_pos_velocity > f7.r_own_neg_velocity).sum()} of {len(f7)}")

**Result: blocked (data missing).** The nasolabialis curve does not bound the claim usefully: 4 of the 5 cut-whisker afferents are anticorrelated with it (mean r -0.277), and only 305A correlates (0.695). These four are better matched by their own positive velocity curve, the variable the paper relates to the intrinsic protractor, whose EMG is not in the dandiset. As a stand-in that is not the criterion, the better of each afferent's correlations with its own positive and negative velocity curves has a mean of 0.753 (range 0.680 to 0.848), near the paper's 0.76. 199B and 209A hold distance-to-pole series and some contact = 1 frames (5,014 and 32) although they were recorded with the whisker cut; at 209A's contact = 1 frames the distance to the pole is 5.6 to 6.5 (1st and 99th percentiles, data check).

## figs4-nasolabialis

> Motor unit activity of m. nasolabialis was phase modulated and correlated with negative whisker
> velocity (r = 0.93) and anti-correlated with positive velocity (r = -0.75) within whisk cycles.

In [ ]:
emg = by_rec["11A"]
w = emg["whisk_phase_curves"]
r_neg = np.corrcoef(w["rate"], w["vel_neg"])[0, 1]
r_pos = np.corrcoef(w["rate"], w["vel_pos"])[0, 1]
print(f"recording 11A ({emg['identifier']}, {emg['subject_id']}): {emg['n_spikes']} spike times, whisking {emg['category_seconds']['whisking'] / 60:.1f} min")
print(f"motor unit rate by phase: {w['rate'].min():.1f} to {w['rate'].max():.1f} Hz, peak at phase {w['centres'][np.argmax(w['rate'])]:.2f} rad, "
      f"where the mean velocity is {'negative' if w['vel_neg'][np.argmax(w['rate'])] > w['vel_pos'][np.argmax(w['rate'])] else 'positive'}")
print(f"r with negative velocity {r_neg:.3f} (paper 0.93, tolerance 0.093); with positive velocity {r_pos:.3f} (paper -0.75, tolerance 0.075)")
print("within tolerance:", abs(r_neg - 0.93) <= 0.093, abs(r_pos + 0.75) <= 0.075)
print("Other EMG files (7A, 8A, 9A) have spike times:", [by_rec[x]["has_spikes"] for x in ("7A", "8A", "9A")],
      "; acquisition groups:", [by_rec[x]["acquisition"] for x in ("7A", "8A", "9A")])
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(w["centres"], w["rate"] / w["rate"].max(), label="motor units")
ax.plot(w["centres"], w["vel_neg"] / w["vel_neg"].max(), label="-velocity")
ax.plot(w["centres"], w["vel_pos"] / w["vel_pos"].max(), label="+velocity"); ax.legend(fontsize=7); ax.set_xlabel("phase (rad)")
plt.show()

**Result: reproduced.** Across 30 phase bins the motor unit spikes of recording 11A follow negative velocity (r = 0.924; paper 0.93) and anti-correlate with positive velocity (r = -0.716; paper -0.75), both within tolerance. The other three EMG files hold no EMG signal and no spike times.

## sigma-kappa

> The noise level of curvature estimates was 7.8 plus or minus 5.0% (mean and SD) across 33
> sessions.

In [ ]:
sk = pd.DataFrame([{"rec": r["rec"], "a": r["sigma_kappa"]["a"], "b": r["sigma_kappa"]["b"]} for r in touch])
print(sk.round(2).to_string(index=False))
for k, lab in (("a", "(a) contact = 1 and distance < 1 mm"), ("b", "(b) contact = 1")):
    print(f"{lab}: mean {sk[k].mean():.2f}%, SD {sk[k].std():.2f}, range {sk[k].min():.2f} to {sk[k].max():.2f} (paper 7.8 and 5.0)")

**Result: reproduced.** The mean curvature noise over the 33 touch recordings is 10.01% with SD 6.97 (paper 7.8 plus or minus 5.0), within the paper's spread but above its mean; two recordings are high, 243B (39.89%) and 265A (28.49%). Both readings of contact frames give the same values.

## Summary

| Claim | Outcome |
|---|---|
| table-s1-link | reproduced |
| video-frames | reproduced |
| merkel-minutes (outside labels) | reproduced |
| fig2c-baseline-rate | reproduced |
| fig2c-touch-rate | reproduced |
| fig2c-wt-star | reproduced (reading b) |
| fig3a-direction-selective (outside labels) | partially reproduced |
| fig3d-gam-full | reproduced |
| fig3d-gam-single | reproduced |
| fig3d-gam-best-pair | partially reproduced |
| fig4c-viscoelastic (outside labels) | reproduced |
| fig5-phase-modulation | reproduced |
| fig5e-phase-span | partially reproduced |
| figs3-phase-invariance | reproduced |
| figs6-kinematic-gam | reproduced (15 of 28 afferents) |
| figs4-nasolabialis | reproduced |
| sigma-kappa | reproduced |
| fig7g-phase-span-28, fig6-cut-eliminated, fig6-cut-remained, fig7e-lost-rate-kinematics | blocked: metadata insufficient |
| fig6a-final-length, fig7d-cut-emg, fig7f-precut-emg, figs4-intrinsic-protractor, figs1-optotag-latency, figs1-labelling-specificity, contact-classifier | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The paper's GAMs (mgcv, 1 ms bins, UBRE smoothing) are replaced by penalized B-spline models fitted as binomial models with two trials per 2 ms frame, on at most 60,000 frames per recording; the viscoelastic model is fitted with L-BFGS-B from eight starts. Confidence intervals for the Glossary classes are bootstrapped over trials. The kinematic models of Figure S6 are tested on 15 of the paper's 28 afferents. Which recording belongs to which experiment, and the afferent types, come from Table S1. Two outcomes sit near their thresholds: the Rayleigh p of fig5e-phase-span moves across 0.05 with the preferred-phase estimator, and M0 and M0' rank second of the 45 pairs, 0.01 behind M0 and velocity, under every variant tried.

**What was found about the dandiset.** The 60 files match Table S1 one to one, and most values come close to the paper's frame counts, rates and model performance. The exceptions are the median non-preferred phase rate (16% above the paper's), the mean curvature noise (28% above), and direction selectivity (65% of Merkel and SA afferents at an index of 0.5 or more, against the 80% the criterion asks). But the files do not say which experiment each recording belongs to, which three touch recordings the paper excluded, the afferent type or Merkel identity, or the meaning of their series (every series is "no description"). The progressive cutting recordings label every trial intact, though their rates fall over the session. The intrinsic protractor EMG, the optogenetic tagging data and the whisker cut lengths are not in the dandiset. The derivatives are stored per millisecond but labelled per second, the stored whisk phase is the negative of the Hilbert phase, so it runs backwards in time, trial intervals are 7 s for 5 s of data, one file's spike times are unsorted, and two identifiers lack the unit letter. Two recordings made with the whisker cut (199B, 209A) hold distance-to-pole series and some contact frames, which the files do not explain.

In [ ]:
print(f"total run time {(time.time() - started) / 60:.1f} min")